# Positional Encoding and Add & Norm
Pure NumPy implementation of sinusoidal positional encoding, layer normalization, and residual Add & Norm.

In [1]:
import numpy as np

seq_len, d_model = 4, 8
x = np.array([[1., 2., 3., 4., 5., 6., 7., 8.]] * seq_len)
sublayer_out = np.ones((seq_len, d_model)) * 0.5
gamma = np.ones(d_model)
beta = np.zeros(d_model)

In [2]:
def positional_encoding(seq_len, d_model):
    positions = np.arange(seq_len)[:, np.newaxis]
    pair_indices = np.arange(0, d_model, 2)
    angles = positions / (10000 ** (pair_indices / d_model))

    pe = np.zeros((seq_len, d_model), dtype=float)
    pe[:, 0::2] = np.sin(angles)
    pe[:, 1::2] = np.cos(angles[:, :pe[:, 1::2].shape[1]])
    return pe

In [3]:
def layer_norm(x, gamma, beta, eps=1e-5):
    mean = np.mean(x, axis=-1, keepdims=True)
    variance = np.var(x, axis=-1, keepdims=True)
    xhat = (x - mean) / np.sqrt(variance + eps)
    return gamma * xhat + beta

In [4]:
def add_and_norm(x, sublayer_out, gamma, beta, eps=1e-5):
    return layer_norm(x + sublayer_out, gamma, beta, eps)

In [5]:
pe = positional_encoding(4, 8)
print(np.round(pe[0], 3))

y = layer_norm(x, np.ones(8), np.zeros(8))
print(np.round(y.mean(axis=-1), 6))
print(np.round(y.var(axis=-1), 4))

z = add_and_norm(x, sublayer_out, np.ones(8), np.zeros(8))
print(np.allclose(z, layer_norm(x + sublayer_out, np.ones(8), np.zeros(8))))

[0. 1. 0. 1. 0. 1. 0. 1.]
[0. 0. 0. 0.]
[1. 1. 1. 1.]
True
